# CI RAG Pipeline — Advanced Modular RAG with PubMedBERT + FAISS + BM25

In [1]:
%load_ext autoreload
%autoreload 2

import pathlib
import sys
import os

import anthropic
import pandas as pd
from IPython.display import display, Markdown

_here = pathlib.Path().resolve()
if (_here / 'clinical-trial-optimizer').exists():
    CT_ROOT = _here / 'clinical-trial-optimizer'
elif (_here / 'src').exists():
    CT_ROOT = _here
elif (_here.parent / 'src').exists():
    CT_ROOT = _here.parent
else:
    raise RuntimeError(f'Cannot locate clinical-trial-optimizer root from {_here}')

sys.path.insert(0, str(CT_ROOT / 'src'))
sys.path.insert(0, str(CT_ROOT / 'src' / 'rag'))

from ct_api import fetch_competing_trials
from rag.chunker import chunk_competing_trials, parse_criteria_via_llm
from rag.embedder import load_pubmedbert, embed_chunks
from rag.index import build_index, load_index
from rag.retriever import retrieve
from rag.retrieval_evaluator import evaluate_retrieval
from rag.ci_reasoner import reason_about_criterion
from rag.recommendation_evaluator import evaluate_recommendations, format_report
from rag.ci_rag_pipeline import run_ci_rag

print(f'CT_ROOT : {CT_ROOT}')

/Users/rajeevkulkarni/ml-explorations/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CT_ROOT : /Users/rajeevkulkarni/ml-explorations/clinical-trial-optimizer


In [2]:
print('API key set' if os.environ.get('ANTHROPIC_API_KEY') else 'WARNING: ANTHROPIC_API_KEY not set')

API key set


In [3]:
# Paste I/E criteria here
ie_criteria = """
Inclusion Criteria:
- Participant must have at least 1 measurable lesion, according to Response Evaluation Criteria in Solid Tumors (RECIST) version 1.1, that has not been previously irradiated
- Participant must have histologically or cytologically confirmed, locally advanced or metastatic, non-squamous non-small cell lung cancer (NSCLC), characterized at or after the time of locally advanced or metastatic disease diagnosis by either epidermal growth factor receptor (EGFR) Exon 19del or Exon 21 L858R mutation
- A participant with a history of brain metastases must have had all lesions treated as clinically indicated (that is, no current indication for further definitive local therapy). Any definitive local therapy to brain metastases must have been completed at least 14 days prior to randomization and the participant can be receiving no greater than10 milligrams (mg) prednisone or equivalent daily for the treatment of intracranial disease
- Participant must have Eastern Cooperative Oncology Group (ECOG) status of 0 or 1
- Any toxicities from prior systemic anticancer therapy must have resolved to National Cancer Institute Common Terminology Criteria for Adverse Events (NCI CTCAE) Version 5.0 Grade 1 or baseline level (except for alopecia [any grade], Grade <= 2 peripheral neuropathy, or Grade <= 2 hypothyroidism stable on hormone replacement)
- A participant of childbearing potential must have a negative serum pregnancy test at screening and within 72 hours of the first dose of study treatment and must agree to further serum or urine pregnancy tests during the study
- Participant must have progressed on or after osimertinib monotherapy as the most recent line of treatment. Osimertinib must have been administered as either the first-line treatment for locally advanced or metastatic disease or in the second- line setting after prior treatment with first- or second-generation EGFR tyrosine kinase inhibitor (TKI) as a monotherapy. Participants who received either neoadjuvant and/or adjuvant treatment of any type are eligible if progression to locally advanced or metastatic disease occurred at least 12 months after the last dose of such therapy and then the participant progressed on or after osimertinib in the locally advanced or metastatic setting. Treatment with osimertinib must be discontinued at least 8 days (4 half-lives) prior to randomization (that is last dose no later than Day -8)

Exclusion Criteria:
- Participant received radiotherapy for palliative treatment of NSCLC less than 14 days prior to randomization
- Participant with symptomatic or progressive brain metastases
- Participant has history of or current evidence of leptomeningeal disease, or participant has spinal cord compression not definitively treated with surgery or radiation
- Participant has known small cell transformation
- Participant has a medical history of interstitial lung disease (ILD), including drug-induced ILD or radiation pneumonitis
- Participant has a history of clinically significant cardiovascular disease including, but not limited to diagnosis of deep vein thrombosis or pulmonary embolism within 4 weeks prior to randomization; myocardial infarction; unstable angina; stroke; transient ischemic attack; coronary/peripheral artery bypass graft; or acute coronary syndrome. Participant has a significant genetic predisposition to venous thromboembolic events. Participant has a prior history of venous thromboembolic events and is not on appropriate therapeutic anticoagulation as per National Comprehensive Cancer Network or local guidelines"
"""

print(f'Length of I/E criteria: {len(ie_criteria)} characters')

Length of I/E criteria: 3585 characters


In [4]:
# ⚠️ HUMAN APPROVAL REQUIRED — live ClinicalTrials.gov API call
competing_trials_md = fetch_competing_trials(ie_criteria)
print(f'Trials retrieved: {competing_trials_md.count("NCT")} trials')

Search terms extracted: condition='NSCLC EGFR', intr='osimertinib'
Trials retrieved: 21 trials


In [5]:
# ⚠️ HUMAN APPROVAL REQUIRED — LLM call per trial
competitor_chunks = chunk_competing_trials(competing_trials_md)
print(f'Total chunks: {len(competitor_chunks)}')
treatment_chunks = [c for c in competitor_chunks if c.metadata.get("treatment_related", False)]
print(f'Treatment-related chunks: {len(treatment_chunks)}')
trials_represented = len(set(c.source_nct_id for c in competitor_chunks if c.source_nct_id))
print(f'Trials represented: {trials_represented}')

Chunking trial 1: NCT02317016
Chunking trial 2: NCT04181060
Chunking trial 3: NCT03969823
Chunking trial 4: NCT06838273
Chunking trial 5: NCT05256290
Chunking trial 6: NCT05120349
Chunking trial 7: NCT02157883
Chunking trial 8: NCT07100080
Chunking trial 9: NCT05382728
Chunking trial 10: NCT07295821
Chunking trial 11: NCT06417814
Chunking trial 12: NCT01802632
Chunking trial 13: NCT04324164
Chunking trial 14: NCT05364073
Chunking trial 15: NCT03239340


Failed to extract criteria for NCT03239340: 'str' object has no attribute 'get'


Chunking trial 16: NCT05805631
Chunking trial 17: NCT03040973
Chunking trial 18: NCT05163249
Chunking trial 19: NCT07028489
Chunking trial 20: NCT01994057
Total chunks: 85
Treatment-related chunks: 21
Trials represented: 19


In [6]:
# Load PubMedBERT and embed all chunks
model = load_pubmedbert()
embeddings = embed_chunks(competitor_chunks, model)
print(f'Embeddings shape: {embeddings.shape}')

# Build hybrid index (FAISS + BM25)
PERSIST_DIR = CT_ROOT / 'data' / 'indexes'
PERSIST_DIR.mkdir(parents=True, exist_ok=True)
hybrid_index = build_index(competitor_chunks, embeddings, 'mariposa2_ci', str(PERSIST_DIR))
print(f'Index built: {hybrid_index.faiss_index.ntotal} vectors indexed')

Loading PubMedBERT...


No sentence-transformers model found with name microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1537.89it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationshi

Embeddings shape: (85, 768)
Index built: 85 vectors indexed


In [8]:
source_chunks = parse_criteria_via_llm(ie_criteria, "SOURCE", "Source Trial", {})
print(f"Total source criteria: {len(source_chunks)}")
if source_chunks:
    test_criterion = source_chunks[0]
    print(f"\nTest criterion: {test_criterion.text[:200]}")
    results = retrieve(test_criterion.text, hybrid_index, model, k=5)
    print(f"\nTop 5 retrieved chunks:")
    for r in results:
        print(f"  [{r.rrf_score:.4f}] {r.chunk.source_nct_id} ({r.chunk.criterion_type}): {r.chunk.text[:100]}")

Total source criteria: 13

Test criterion: Participant must have at least 1 measurable lesion, according to Response Evaluation Criteria in Solid Tumors (RECIST) version 1.1, that has not been previously irradiated

Top 5 retrieved chunks:
  [0.0328] NCT05382728 (inclusion): At least one measurable lesion according to Response Evaluation Criteria in Solid Tumours (RECIST) v
  [0.0315] NCT06838273 (inclusion): At least one measurable lesion meeting the RECIST v1.1 definition
  [0.0312] NCT05364073 (inclusion): Disease that has progressed after at least one available standard therapy; or for whom standard ther
  [0.0161] NCT04181060 (inclusion): Patient must have somatic activating sensitizing mutation in EGFR (e.g. but not limited to Exon)
  [0.0159] NCT07295821 (inclusion): Participants with histologically documented NSCLC of predominantly nonsquamous, squamous, and adenos


In [10]:
# ⚠️ HUMAN APPROVAL REQUIRED — Claude Sonnet API call
if treatment_source:
    recommendation = reason_about_criterion(test_criterion.text, results)
    print(f'Criterion: {recommendation.criterion_text[:150]}')
    print(f'Label: {recommendation.label}')
    print(f'Confidence: {recommendation.confidence:.2f}')
    print(f'Rationale: {recommendation.rationale}')
    print(f'Evidence trials: {recommendation.evidence_trials}')
    if recommendation.suggested_wording != test_criterion.text:
        print(f'Suggested wording: {recommendation.suggested_wording}')

Criterion: Participant must have at least 1 measurable lesion, according to Response Evaluation Criteria in Solid Tumors (RECIST) version 1.1, that has not been 
Label: KEEP
Confidence: 0.72
Rationale: The source criterion aligns with standard practice observed in competing trials such as NCT05382728 and NCT06838273, both of which require at least one measurable lesion per RECIST v1.1 without additional qualifiers. The added specification that the lesion must not have been previously irradiated is a clinically meaningful and common safeguard to ensure accurate response assessment, and its absence in competing trials does not suggest it should be removed—rather, it reflects a more rigorous standard that protects measurement integrity.
Evidence trials: ['NCT05382728', 'NCT06838273']


In [12]:
# ⚠️ HUMAN APPROVAL REQUIRED — full pipeline run
print('Running full CI RAG pipeline...')
results_df = run_ci_rag(
    ie_criteria,
    competing_trials_md,
    persist_dir=str(PERSIST_DIR),
)
print(f'\nResults: {len(results_df)} criteria evaluated')
display(results_df[['criterion_type', 'criterion_text', 'label', 'confidence', 'rationale']].head(10))

Running full CI RAG pipeline...
Chunking trial 1: NCT02317016
Chunking trial 2: NCT04181060
Chunking trial 3: NCT03969823
Chunking trial 4: NCT06838273
Chunking trial 5: NCT05256290
Chunking trial 6: NCT05120349
Chunking trial 7: NCT02157883
Chunking trial 8: NCT07100080
Chunking trial 9: NCT05382728
Chunking trial 10: NCT07295821
Chunking trial 11: NCT06417814
Chunking trial 12: NCT01802632
Chunking trial 13: NCT04324164
Chunking trial 14: NCT05364073
Chunking trial 15: NCT03239340
Chunking trial 16: NCT05805631
Chunking trial 17: NCT03040973
Chunking trial 18: NCT05163249
Chunking trial 19: NCT07028489
Chunking trial 20: NCT01994057
Loading PubMedBERT...


No sentence-transformers model found with name microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1605.98it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationshi


Results: 13 criteria evaluated


,criterion_type,criterion_text,label,confidence,rationale
0,inclusion,Participant must have at least 1 measurable le...,KEEP,0.78,The requirement for at least one measurable le...
1,inclusion,Participant must have histologically or cytolo...,KEEP,0.78,The source criterion aligns well with the prev...
2,inclusion,A participant with a history of brain metastas...,KEEP,0.30,The competing trials provided have very low re...
3,inclusion,Participant must have Eastern Cooperative Onco...,RELAX,0.62,Multiple competing trials in similar EGFR-muta...
4,inclusion,Any toxicities from prior systemic anticancer ...,KEEP,0.40,The source criterion reflects a well-establish...
5,inclusion,A participant of childbearing potential must h...,KEEP,0.55,The source criterion requiring a negative seru...
6,inclusion,Participant must have progressed on or after o...,KEEP,0.62,The source criterion is highly specific and we...
7,exclusion,Participant received radiotherapy for palliati...,KEEP,0.42,The 14-day washout period for palliative radio...
8,exclusion,Participant with symptomatic or progressive br...,KEEP,0.45,The source criterion excluding participants wi...
9,exclusion,Participant has history of or current evidence...,KEEP,0.35,The source criterion excluding participants wi...
